In [2]:

import numpy as np # linear algebra
import pandas as pd # data processing, CSV file I/O (e.g. pd.read_csv)
import os
for dirname, _, filenames in os.walk('/kaggle/input'):
    for filename in filenames:
        print(os.path.join(dirname, filename))

import kagglehub
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
from sklearn.model_selection import TimeSeriesSplit, GridSearchCV
from sklearn.neighbors import KNeighborsClassifier
from sklearn.ensemble import RandomForestClassifier
from sklearn.svm import SVC
from sklearn.preprocessing import RobustScaler
from sklearn.metrics import average_precision_score, precision_recall_curve
from sklearn.utils import resample

/kaggle/input/datasets/amanalisiddiqui/fraud-detection-dataset/AIML Dataset.csv
/kaggle/input/competitions/ieee-fraud-detection/sample_submission.csv
/kaggle/input/competitions/ieee-fraud-detection/test_identity.csv
/kaggle/input/competitions/ieee-fraud-detection/train_identity.csv
/kaggle/input/competitions/ieee-fraud-detection/test_transaction.csv
/kaggle/input/competitions/ieee-fraud-detection/train_transaction.csv


In [3]:
df_trans = pd.read_csv('/kaggle/input/competitions/ieee-fraud-detection/train_transaction.csv')
df_id = pd.read_csv('/kaggle/input/competitions/ieee-fraud-detection/train_identity.csv')
# preserve all transactions
df = df_trans.merge(df_id, on='TransactionID', how='left')

**DATASET**

In [4]:
print(df.head())

   TransactionID  isFraud  TransactionDT  TransactionAmt ProductCD  card1  \
0        2987000        0          86400            68.5         W  13926   
1        2987001        0          86401            29.0         W   2755   
2        2987002        0          86469            59.0         W   4663   
3        2987003        0          86499            50.0         W  18132   
4        2987004        0          86506            50.0         H   4497   

   card2  card3       card4  card5  ...                id_31  id_32  \
0    NaN  150.0    discover  142.0  ...                  NaN    NaN   
1  404.0  150.0  mastercard  102.0  ...                  NaN    NaN   
2  490.0  150.0        visa  166.0  ...                  NaN    NaN   
3  567.0  150.0  mastercard  117.0  ...                  NaN    NaN   
4  514.0  150.0  mastercard  102.0  ...  samsung browser 6.2   32.0   

       id_33           id_34  id_35 id_36 id_37  id_38  DeviceType  \
0        NaN             NaN    NaN   Na

In [5]:
print(df.columns.tolist)
print(f"Size of transaction dataset: {df_trans.memory_usage(deep=True).sum() / 1024**2:.2f} MB")
print(f"Size of identity dataset: {df_id.memory_usage(deep=True).sum() / 1024**2:.2f} MB")

<bound method IndexOpsMixin.tolist of Index(['TransactionID', 'isFraud', 'TransactionDT', 'TransactionAmt',
       'ProductCD', 'card1', 'card2', 'card3', 'card4', 'card5',
       ...
       'id_31', 'id_32', 'id_33', 'id_34', 'id_35', 'id_36', 'id_37', 'id_38',
       'DeviceType', 'DeviceInfo'],
      dtype='object', length=434)>
Size of transaction dataset: 2062.07 MB
Size of identity dataset: 143.14 MB


**EDA**

In [6]:
print(df['isFraud'].value_counts(normalize=True))

isFraud
0    0.96501
1    0.03499
Name: proportion, dtype: float64


In [7]:
dtype_df = pd.DataFrame(df.dtypes.reset_index())
dtype_df.columns = ['Column', 'Data Type']
print(dtype_df)

             Column Data Type
0     TransactionID     int64
1           isFraud     int64
2     TransactionDT     int64
3    TransactionAmt   float64
4         ProductCD    object
..              ...       ...
429           id_36    object
430           id_37    object
431           id_38    object
432      DeviceType    object
433      DeviceInfo    object

[434 rows x 2 columns]


In [8]:
print(df.dtypes.value_counts())

float64    399
object      31
int64        4
Name: count, dtype: int64


In [9]:
results = {}
def find_nans(df, target='isFraud'):
    
    missing_pct = df.isna().mean() * 100
    high_missing = missing_pct[missing_pct > 50].index.tolist()
    results['high_missing_cols'] = high_missing
    print(f"Columns with >50% NaN: {high_missing}")
    
    placeholder_values = ['', 'NULL', 'null', 'None', 'NA', 'N/A', '-1', '-999']
    
    for col in df.select_dtypes(include=['object']).columns:
        for placeholder in placeholder_values:
            if placeholder in df[col].values:
                pct = (df[col] == placeholder).mean() * 100
                if pct > 0:
                    print(f"Column '{col}' has {pct:.1f}% '{placeholder}' values")
    
    return

eda(df)

NameError: name 'eda' is not defined

In [ ]:
cat_cols = df.select_dtypes(include=['object']).columns
fraud_rates = {}  

# proportion of ones
global_fraud_rate = df['isFraud'].mean()

for col in cat_cols:
    fraud_rate = df.groupby(col)['isFraud'].mean().sort_values(ascending=False)
    fraud_rates[col] = fraud_rate
    
    # columns where max fraud rate is bigger than two times
    # the global average
    if fraud_rate.max() > global_fraud_rate * 2:
        print(f"\nFraud rate by {col}:")
        print(fraud_rate.head(5))

In [ ]:
from scipy.stats import ks_2samp

"""
Columns with higher ks statistic show more clear separation
between data points from one class and data points from the other
"""

numeric_cols = df.select_dtypes(include=[np.number]).columns
ks_results = []

#1% of fraud samples OR 30, whichever is larger
min_fraud_samples = max(30, int(fraud_count * 0.01))

for col in numeric_cols:
    fraud_vals = df[df['isFraud'] == 1][col].dropna()
    legit_vals = df[df['isFraud'] == 0][col].dropna()

    """
    The empirical distribution from a small sample is unlikely to 
    represent the true population distribution, 
    leading to unreliable and non-reproducible KS values.

    Therefore, the KS statistic is unstable in that case.
    """
    
    if len(fraud_vals) >= min_fraud_samples and len(legit_vals) >= min_fraud_samples:
        ks_stat = ks_2samp(fraud_vals, legit_vals).statistic
        ks_stat = ks_2samp(fraud_vals, legit_vals).statistic
        ks_results.append((col, ks_stat, len(fraud_vals)))  

ks_df = pd.DataFrame(ks_results, columns=['feature', 'KS', 'fraud_samples'])
top_ks = ks_df.sort_values('KS', ascending=False).head(10)

print(f"\nTop 10 by KS statistic:\n{top_ks}")

In [ ]:

"""
The KS threshold indicates where the cdf's of each class differ the most,
or for which value of the feature they are the most dissimilar
"""
from scipy.stats import ks_2samp

for col in top_ks['feature']:
    fraud_vals = df[df['isFraud'] == 1][col].dropna().sort_values()
    legit_vals = df[df['isFraud'] == 0][col].dropna().sort_values()

    all_vals = np.sort(np.concatenate([fraud_vals.values, legit_vals.values]))
    
    fraud_cdf = np.searchsorted(fraud_vals, all_vals, side='right') / len(fraud_vals)
    legit_cdf = np.searchsorted(legit_vals, all_vals, side='right') / len(legit_vals)
    
    # Find max difference
    diffs = np.abs(fraud_cdf - legit_cdf)
    max_idx = np.argmax(diffs)
    
    threshold_results.append((col, all_vals[max_idx], diffs[max_idx]))

In [ ]:
"""
Columns with higher iv show more clear separation
between data points from one class and data points from the other.
"""
def calculate_iv(df, feature, target):
        """Basic IV calculation for binary classification"""
        grouped = df.groupby(feature)[target].agg(['count', 'sum'])
        grouped['non_fraud'] = grouped['count'] - grouped['sum']
        grouped['pct_fraud'] = grouped['sum'] / grouped['sum'].sum()
        grouped['pct_nonfraud'] = grouped['non_fraud'] / grouped['non_fraud'].sum()
        grouped['woe'] = np.log((grouped['pct_fraud'] + 0.5) / (grouped['pct_nonfraud'] + 0.5))
        grouped['iv'] = (grouped['pct_fraud'] - grouped['pct_nonfraud']) * grouped['woe']
        return grouped['iv'].sum()

iv_results = []
for col in cat_cols:
    iv = calculate_iv(df, col, 'isFraud')
    iv_results.append((col, iv))
iv_df = pd.DataFrame(iv_results, columns=['feature', 'IV'])
results['iv_scores'] = iv_df.sort_values('IV', ascending=False)
print(f"\nInformation Value (all categorical):\n{results['iv_scores']}")

In [ ]:
"""
The IV of each category, before being all aggregated, can show a low percentage of 
datapoints of class A for that category, although the ratio of datapoints of class A 
for that feature to all datapoints of class A is high. This would mean that that category 
has a strong predictive value but given the small sample size, it is unlikely the empirical 
distribution represents the true one.

Theefore a min size sample has to be specified.
"""
def calculate_iv(df, feature, target, mmin_fraud_samples):
    grouped = df.groupby(feature)[target].agg(['count', 'sum'])
    
    
    grouped = grouped[grouped['count'] >= min_fraud_samples]
    
    if len(grouped) == 0:
        return 0
    
    grouped['non_fraud'] = grouped['count'] - grouped['sum']
    grouped['pct_fraud'] = grouped['sum'] / grouped['sum'].sum()
    grouped['pct_nonfraud'] = grouped['non_fraud'] / grouped['non_fraud'].sum()
    grouped['woe'] = np.log((grouped['pct_fraud'] + 0.5) / (grouped['pct_nonfraud'] + 0.5))
    grouped['iv'] = (grouped['pct_fraud'] - grouped['pct_nonfraud']) * grouped['woe']
    return grouped['iv'].sum()

iv_results = []
for col in cat_cols:
    iv = calculate_iv(df, col, 'isFraud', min_fraud_samples)
    iv_results.append((col, iv))
iv_df = pd.DataFrame(iv_results, columns=['feature', 'IV'])
results['iv_scores'] = iv_df.sort_values('IV', ascending=False)
print(f"\nInformation Value (all categorical):\n{results['iv_scores']}")

In [ ]:
"""
If the proabability stability index is high, it indicates the distributions of the samples 
(a sample of datapoints from the past and one from the future) are very different.
It might be possible that the model is assigning categories randomly
"""

if 'TransactionDT' in df.columns:
    df_sorted = df.sort_values('TransactionDT')
    
    def calculate_psi(expected, actual, buckets=10):
        expected_bins = pd.qcut(expected, q=buckets, duplicates='drop')
        expected_counts = expected.groupby(expected_bins).count() / len(expected)
        actual_counts = actual.groupby(expected_bins).count() / len(actual)
        psi = ((actual_counts - expected_counts) * np.log(actual_counts / expected_counts)).sum()
        return psi

    """
    The samples for the calculation of the PSI statistic should not be adjacent, otherwise 
    temporary noise  masks how the data distribution of the classes for each feature drifts
    """
    for i in range(3):  
        expected_start = i * window_size
        expected_end = (i + 1) * window_size
        actual_start = (i + 2) * window_size  
        actual_end = (i + 3) * window_size
        
        if actual_end <= len(df_sorted):
            expected_window = df_sorted.iloc[expected_start:expected_end]
            actual_window = df_sorted.iloc[actual_start:actual_end]
            
            for col in numeric_cols[:10]:
                psi = calculate_psi(expected_window[col].dropna(), actual_window[col].dropna())
                psi_results.append((col, psi, f"window_{i}"))
    
    psi_df = pd.DataFrame(psi_results, columns=['feature', 'PSI', 'comparison'])
    results['high_drift'] = psi_df[psi_df['PSI'] > 0.1]
    print(f"\nFeatures with PSI > 0.1 (time drift):\n{results['high_drift']}")


**TIME-BASED SPLIT**

In [ ]:
df = df.sort_values('TransactionDT')
X = df.drop('isFraud', axis=1)
y = df['isFraud']

"""
For big datasets, n-splits = 10 is a good trafeoff between the size 
of each training set and the number of prediction sets
"""
tscv = TimeSeriesSplit(n_splits=10)

for fold, (train_idx, test_idx) in enumerate(tscv.split(X), 1):
    X_train, X_test = X.iloc[train_idx], X.iloc[test_idx]
    y_train, y_test = y.iloc[train_idx], y.iloc[test_idx]
    print(f"Fold {fold}: Train rows = {len(X_train)}, Test rows = {len(X_test)}, Fraud in test = {y_test.sum()}")

df = df.sort_values('TransactionDT')
X = df.drop('isFraud', axis=1)
y = df['isFraud']

tscv = TimeSeriesSplit(n_splits=10)

for fold, (train_idx, test_idx) in enumerate(tscv.split(X), 1):
    X_train, X_test = X.iloc[train_idx], X.iloc[test_idx]
    y_train, y_test = y.iloc[train_idx], y.iloc[test_idx]
    
print(f"\nTotal test rows across all folds: {sum([len(X.iloc[test_idx]) for _, (_, test_idx) in enumerate(tscv.split(X))])}")
print(f"Average test rows per fold: {sum([len(X.iloc[test_idx]) for _, (_, test_idx) in enumerate(tscv.split(X))]) / tscv.n_splits:.0f}")

**SCALING**

In [ ]:
"""
scaler = RobustScaler()
X_train_scaled = scaler.fit_transform(X_train)
X_test_scaled = scaler.transform(X_test)
"""